In [1]:
# Standar Library
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
# Import library Scikit-Learn
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB

In [3]:
# Import Library Sklearn (Evaluasi Tak Bertingkat)
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix

# Import Library Sklearn (Evaluasi Berperingkat)
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import average_precision_score

In [9]:
# Import Library untuk Stemming
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

ModuleNotFoundError: No module named 'Sastrawi'

In [8]:
# Open Dataset
data = pd.read_excel('dataKumparan1.xlsx')
data.head()

,Topic,Title,Content
0,Politik,"Pelanggaran Pemilu, Tiga Caleg di Sulteng Dipr...","Komisioner Bawaslu Sigi, Sulawesi Tengah, Agus..."
1,Politik,"Pemilu Susulan di Kota Jayapura, Suara Jokowi ...",Walaupun dua dari lima distrik melakukan pemil...
2,Politik,"Tsamara Amany Dipinang, Pengurus PSI Daerah Me...","Tsamara Amany, politisi Partai Solidaritas Ind..."
3,Politik,Ada 47 TPS di Sulawesi Utara Berpotensi Pemili...,Badan Pengawas Pemilu (Bawaslu) Provinsi Sulaw...
4,Politik,Ketua KPPS di Sleman Ditemukan Tewas Gantung D...,"Tugiman, Ketua Kelompok Penyelenggara Pemungut..."


In [9]:
# Ukuran Dataset
print(f"Ukuran Dataset: {data.shape}")

Ukuran Dataset: (60, 3)


In [10]:
# Pembagian data training & testing
x_train, x_test, y_train, y_test = train_test_split(data['Content'], data['Topic'], train_size=0.5, test_size=0.16)


In [11]:
train_data = pd.DataFrame({'teks':x_train, 'label':y_train})
test_data = pd.DataFrame({'teks':x_test, 'label':y_test})

In [12]:
df1 = pd.DataFrame(train_data)
print(df1)

                                                 teks      label
13  Aplikasi dan situs ayojagatps.com ikut meramai...    Politik
36  Platform media sosial populer Instagram kembal...  Teknologi
48  Pariwisata Bintan terus menunjukan perkembanga...     Travel
54  Selain mempertimbangkan budaya, alam, dan kuli...     Travel
1   Walaupun dua dari lima distrik melakukan pemil...    Politik
14  KPU memberikan klarifikasi mengenai banyaknya ...    Politik
19  Cawapres 02 Sandiaga Uno menanggapi rencana ca...    Politik
52  Hari Bumi yang diperingati pada tanggal 22 Apr...     Travel
40  Wahana hiburan Trampolin hadir pertama kali di...     Travel
56  Sulawesi Utara (Sulut) menunjukkan komitmennya...     Travel
29  Mesin pencari Google menampilkan sebuah doodle...  Teknologi
45  Jumat (19/4) kawasan Terminal Keberangkatan Do...     Travel
18  Isu kecurangan di Pemilu 2019 terus menyeruak....    Politik
59  Di balik kemegahan Pegunungan Tianzhu China, a...     Travel
50  Untuk pertama kalinya

In [14]:
# test_data.head()
df2 = pd.DataFrame(test_data)
print(df2)

                                                 teks      label
15  Mantan Ketua Mahkamah Konstitusi (MK), Mahfud ...    Politik
44  Hari Guru atau Minggu Apreasi Guru di Amerika ...     Travel
30  Kemajuan teknologi memudahkan dan mengubah keb...  Teknologi
34  Persaingan aplikasi pesan instan kian sengit s...  Teknologi
55  Maskapai bersimbol singa merah, Lion Air kerap...     Travel
9   Seorang Ketua KPPS bernama Baharuddin Effendi ...    Politik
39  Perusahaan e-commerce marketplace Tokopedia ke...  Teknologi
23  Untuk memberikan pengalaman yang berbeda bagi ...  Teknologi
38  Kamu mungkin pernah merasa kesulitan untuk ber...  Teknologi
57  Kabar bahagia datang bagi para penyelam di sel...     Travel


In [15]:
# Ukuran Data Training & Testing
print(f"Ukuran data train: {train_data.shape}")
print(f"Ukuran data test: {test_data.shape}")

n_train = train_data.shape[0]
n_test = test_data.shape[0]

Ukuran data train: (30, 2)
Ukuran data test: (10, 2)


In [16]:
sparse_data = pd.concat([train_data, test_data], ignore_index=True)
sparse_data.head()
df3 = pd.DataFrame(sparse_data)
print(df3)

                                                 teks      label
0   Aplikasi dan situs ayojagatps.com ikut meramai...    Politik
1   Platform media sosial populer Instagram kembal...  Teknologi
2   Pariwisata Bintan terus menunjukan perkembanga...     Travel
3   Selain mempertimbangkan budaya, alam, dan kuli...     Travel
4   Walaupun dua dari lima distrik melakukan pemil...    Politik
5   KPU memberikan klarifikasi mengenai banyaknya ...    Politik
6   Cawapres 02 Sandiaga Uno menanggapi rencana ca...    Politik
7   Hari Bumi yang diperingati pada tanggal 22 Apr...     Travel
8   Wahana hiburan Trampolin hadir pertama kali di...     Travel
9   Sulawesi Utara (Sulut) menunjukkan komitmennya...     Travel
10  Mesin pencari Google menampilkan sebuah doodle...  Teknologi
11  Jumat (19/4) kawasan Terminal Keberangkatan Do...     Travel
12  Isu kecurangan di Pemilu 2019 terus menyeruak....    Politik
13  Di balik kemegahan Pegunungan Tianzhu China, a...     Travel
14  Untuk pertama kalinya

In [17]:
# Ukuran Sparse Data
print(f"Ukuran sparse data: {sparse_data.shape}")
n_document = sparse_data.shape[0]

Ukuran sparse data: (40, 2)


## Buat daftar stop word dari data yang terdapat dalam file Excel yang telah disediakan.

In [36]:
# Create a stop word list from the Excel data
stop_words = set()
for index, row in data.iterrows():
  if isinstance(row['Content'], str):
    words = row['Content'].lower().split()
    for word in words:
      stop_words.add(word)

print("Stop word list:", stop_words)

Stop word list: {'kunjungi,', 'gugur,', 'rudiantara.', 'menampilkan', 'budaya,', 'tabayun', 'aziz', 'mei.', 'ngurah', 'penggemarnya', 'ditutup', 'petualangan,', 'dekat', 'sudarno', 'mengubah', 'nah', 'aparat', 'diembannya.', 'surat', '“ada', 'kelaikudaraan', 'komisinya', 'swipe', 'dilaksanakan.', 'berbeda,', 'berjam-jam.', 'laguna', 'ditarik', 'informasi.', 'maret,', 'jam,', 'mewakili', 'mengalihkan', 'per', 'bikin', '1958.', '"ya', 'unggu', '94', 'salib', 'sandi.', 'setuju,', 'setiap', 'river.', 'melanjutkan', 'kian', 'peserta', 'terbaru', 'ayam,', 'timur,', 'merekam', 'dulu', 'redaksi', 'pemberian', 'fisik', 'usai.', 'superman', 'instansi:', 'tercapai,', 'guru', 'noda,', 'lurah,', 'komitmen', 'masukkan', 'beyond', 'bali.', 'menit.', 'djoko', 'penjumlahan', 'potensial', 'surga', 'diadakan', 'ktp', 'rekayasa', 'saifudin,', 'andy.', 'permasalahan', 'besoknya', '“apa', 'berminat', 'mengakui', 'smartphone', 'abad', 'eiffel.', 'temuan', 'segera', 'asdep', '6s,', 'bolmunt', '(24/4)', 'dpr',

## Skenario 2 - Tanpa Stemming Tanpa StopWord

## Skenario 3 - Dengan Stemming dengan StopWord

**Stemming**

In [21]:
# Create stemmer
stemmerFactory = StemmerFactory()
stemmer = stemmerFactory.create_stemmer()

# Stem Process
for row in range(n_document):
    sparse_data.loc[row, 'teks'] = stemmer.stem(sparse_data.loc[row, 'teks'])

In [22]:
df4 = pd.DataFrame(sparse_data)
print(df4)

                                                 teks      label
0   aplikasi dan situs ayojagatps com ikut ramai h...    Politik
1   platform media sosial populer instagram kembal...  Teknologi
2   pariwisata bintan terus tunjuk kembang yang po...     Travel
3   selain timbang budaya alam dan kuliner pasu fa...     Travel
4   walaupun dua dari lima distrik laku pilih umum...    Politik
5   kpu beri klarifikasi kena banyak temu salah in...    Politik
6   cawapres 02 sandiaga uno tanggap rencana cawap...    Politik
7   hari bumi yang ingat pada tanggal 22 april bua...     Travel
8   wahana hibur trampolin hadir pertama kali di k...     Travel
9   sulawesi utara sulut tunjuk komitmen untuk kem...     Travel
10  mesin cari google tampil buah doodle atau gamb...  Teknologi
11  jumat 19 4 kawasan terminal berangkat domestik...     Travel
12  isu curang di milu 2019 terus seruak dua timse...    Politik
13  di balik megah gunung tianzhu china ada buah d...     Travel
14  untuk pertama kali da

In [23]:
vectorizer = CountVectorizer()
tf = vectorizer.fit_transform(sparse_data['teks'])

print(f"Jumlah dokumen: {tf.shape[0]}")
print(f"Jumlah term: {tf.shape[1]}")

Jumlah dokumen: 40
Jumlah term: 3005


In [24]:
print("Daftar Term:")
vectorizer.get_feature_names_out()

Daftar Term:


array(['00', '000', '0004', ..., 'zat', 'ziarah', 'zoetry'], dtype=object)

In [27]:
print("Daftar Stopword")
vectorizer.get_stop_words()

Daftar Stopword


In [30]:
print("Matriks Tf:")
tf_matrix = pd.DataFrame(tf.toarray(), columns=vectorizer.get_feature_names_out()) # Corrected keyword argument to 'columns'
tf_matrix

Matriks Tf:


,00,000,0004,01,02,039,04,043,052,056,...,yakni,yang,yerusalem,yesus,yogyakarta,yunani,yusuf,zat,ziarah,zoetry
0,1,0,0,1,1,0,0,0,1,0,...,1,15,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,7,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,1,14,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,9,0,0,0,0,0,0,0,0
4,0,0,0,1,2,1,0,2,0,1,...,0,3,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,...,0,10,0,0,0,0,0,0,0,0
6,0,0,0,1,1,0,0,0,0,0,...,0,8,0,0,1,0,1,0,0,0
7,0,0,0,0,0,0,0,0,0,0,...,0,11,0,0,0,0,0,0,0,0
8,2,1,0,0,0,0,0,0,0,0,...,0,4,0,0,0,0,0,0,0,0
9,0,0,0,0,0,0,0,0,0,0,...,0,5,0,0,0,0,0,0,0,0


In [31]:
print("Matriks Tf (Khusus data train:)")
tf_train = tf_matrix[:n_train]
tf_train.shape

Matriks Tf (Khusus data train:)


(30, 3005)

In [32]:
# Penyesuain df agar query(data set) tidak dihitung pada perhitungan df
transformer = TfidfTransformer(sublinear_tf=True)
n = n_train
df = tf_train.astype(bool).sum(axis=0)
idf = np.log(n/df)
transformer.idf_ = idf

weight = transformer.fit_transform(tf)
print(f"Jumlah Dokumen: {weight.shape[0]}")
print(f"Jumlah Term: {weight.shape[1]}")

Jumlah Dokumen: 40
Jumlah Term: 3005


In [33]:
weight_matrix = pd.DataFrame(weight.toarray(), columns=vectorizer.get_feature_names_out())
weight_matrix

,00,000,0004,01,02,039,04,043,052,056,...,yakni,yang,yerusalem,yesus,yogyakarta,yunani,yusuf,zat,ziarah,zoetry
0,0.050268,0.000000,0.000000,0.050268,0.050268,0.000000,0.000000,0.000000,0.060739,0.000000,...,0.044142,0.056020,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000
1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.065951,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000
2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.048344,0.060211,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000
3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.052630,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000
4,0.000000,0.000000,0.000000,0.054773,0.092739,0.066184,0.000000,0.112059,0.000000,0.066184,...,0.000000,0.034547,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000
5,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.065115,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000
6,0.000000,0.000000,0.000000,0.060247,0.060247,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.055759,0.000000,0.00000,0.065456,0.000000,0.072798,0.000000,0.00000,0.000000
7,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.063048,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000
8,0.109944,0.060580,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.046571,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000
9,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.043475,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000


In [34]:
# Pembagian matriks bobot
weight_train = weight_matrix[:n_train]
weight_test = weight_matrix[n_train:]

If **file code & images** cannot be accessed, you can use the repository link which can be accessed by using [this link!](https://colab.research.google.com/drive/1Te6z5clvOH31d2NwWRjT4HeUgiUwFI89?usp=sharing "link Google Colab")

## THANK YOU😸

<img src="https://i.pinimg.com/originals/47/20/51/472051c6d88bd2837525520597251d67.gif" alt="Footer Background">